# SimMOF End-to-End Demo

This notebook demonstrates how SimMOF converts natural-language MOF simulation requests into structured workflows.

We will cover:
1. a simple end-to-end example,
2. a query that requires clarification,
3. patching user-provided simulation input.

In [1]:
import json

from query.agent import analyze_mof_query
from working.agent import WorkingAgent

from Zeopp.agent import ZeoppAgent
from LAMMPS.agent import LAMMPSAgent
from response.agent import ResponseAgent
from RASPA.agent import RASPAAgent
from VASP.agent import VASPAgent
from screening.agent import ScreeningAgent
from analysis.agent import AnalysisAgent

from main import patch_simulation_input_with_user_reply
from config import LLM_DEFAULT

/home/users/skyljw0714/anaconda3/envs/simmof/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
agents = {
    "ZeoppAgent": ZeoppAgent(),
    "LAMMPSAgent": LAMMPSAgent(),
    "ResponseAgent": ResponseAgent(),
    "RASPAAgent": RASPAAgent(),
    "VASPAgent": VASPAgent(),
    "ScreeningAgent": ScreeningAgent(),
    "AnalysisAgent": AnalysisAgent(),
}

In [3]:
def show_result(title, obj):
    print(f"\n{'='*80}")
    print(title)
    print(f"{'='*80}")
    if isinstance(obj, str):
        print(obj)
    else:
        print(json.dumps(obj, ensure_ascii=False, indent=2))


def build_workflow(user_query, agents):
    bundle = analyze_mof_query(user_query)

    if not bundle or not bundle.get("queries"):
        return {
            "ok": False,
            "error": "Query parsing failed.",
            "bundle": bundle,
        }

    parsed = bundle["queries"]
    analysis_enabled = bundle.get("analysis_enabled", False)
    simulation_input = bundle.get("simulation_input")

    wa = WorkingAgent(
        parsed,
        analysis_enabled=analysis_enabled,
        agents=agents,
        simulation_input=simulation_input,
    )

    plans = wa.plan()

    return {
        "ok": True,
        "bundle": bundle,
        "parsed": parsed,
        "analysis_enabled": analysis_enabled,
        "simulation_input": simulation_input,
        "plans": [p.model_dump() for p in plans],
        "worker": wa,
    }

## How to read this notebook

For each example, we will inspect:
- the original user query,
- the parsed query bundle,
- whether clarification is needed,
- the planned workflow,
- and optionally the execution behavior.

## Example 1 — Simple Zeo++ workflow

In this example, the user asks for the surface area of UiO-66.

This is a simple structural-property query that should map naturally to a Zeo++-based workflow,
making it a good introductory example for new users.

In [4]:
user_query_1 = "I want to calculate the surface area of UiO-66"
print(user_query_1)

I want to calculate the surface area of UiO-66


In [5]:
demo1 = build_workflow(user_query_1, agents)

show_result("Parsed bundle", demo1["bundle"])
show_result("Parsed queries", demo1["parsed"])
show_result("Planned workflows", demo1["plans"])

=== Parsed Queries ===
- UiO-66-surface_area: ZeoppAgent → surface_area (UiO-66, guest=None)

Parsed bundle
{
  "queries": [
    {
      "Name": "UiO-66-surface_area",
      "Agent": "ZeoppAgent",
      "Property": "surface_area",
      "MOF": "UiO-66",
      "Guest": null,
      "QueryText": "I want to calculate the surface area of UiO-66"
    }
  ],
  "analysis_enabled": false,
  "simulation_input": {
    "present": false,
    "snippets": []
  },
  "simulation_input_status": "ok",
  "simulation_input_message": "",
  "needs_clarification": false,
  "missing_fields": [],
  "clarification_question": ""
}

Parsed queries
[
  {
    "Name": "UiO-66-surface_area",
    "Agent": "ZeoppAgent",
    "Property": "surface_area",
    "MOF": "UiO-66",
    "Guest": null,
    "QueryText": "I want to calculate the surface area of UiO-66"
  }
]

Planned workflows
[
  {
    "job_name": "UiO-66_surface_area",
    "agent": "ZeoppAgent",
    "mof": "UiO-66",
    "guest": null,
    "property": "surface_area"

In [6]:
# Uncomment the lines below if you want to run the full workflow.
# Depending on your environment, this may trigger the actual Zeo++-based execution.

results_1 = await demo1["worker"].run_async()
show_result("Execution results", results_1)

Saving output to: /home/users/skyljw0714/SimMOF/working_dir/UiO-66_surface_area
Successfully fetched UiO-66 structure
[MOFCHECKER] Validation passed: UiO-66.cif


/home/users/skyljw0714/anaconda3/envs/simmof/lib/python3.9/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 10020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


[RAG] no relevant Zeopp hints
[ZeoppInputAgent] Zeo++ command: /home/users/skyljw0714/SimMOF/Zeopp/network -ha -sa 1.2 1.2 50000 /home/users/skyljw0714/SimMOF/working_dir/UiO-66_surface_area/UiO-66.cif
[ZeoppErrorAgent] zeopp_status == ok → nothing to do.

=== ResponseAgent: final user-facing answer ===

We computed the accessible surface area (ASA) of UiO-66 (empty framework, no guest) using Zeo++ with its default high-accuracy settings. 
Result: ASA = 2149.93 m²/g and 2582.51 m²/cm³. 
For completeness, the raw cell-based ASA reported by Zeo++ is 594.078 Å². 
The Voronoi analysis identified 1 continuous channel and no isolated pockets, indicating a percolated pore network for the chosen probe. 
If you need results for a specific probe radius or a BET-style comparison, I can rerun accordingly.

=== End of Response ===


Execution results
{
  "UiO-66_surface_area": {
    "UiO-66_surface_area_job": {
      "plan_name": "UiO-66_surface_area",
      "job_name": "UiO-66_surface_area",
     

## Example 2 — Query requiring clarification

Now we test a more ambiguous request:

`Calculate adsorption in HKUST-1`

This query is incomplete because it does not specify key simulation conditions such as:
- the adsorbate species,
- temperature,
- pressure,
- or the exact adsorption-related property of interest.

In this example, we show how SimMOF detects the missing information,
asks for clarification, and then replans the workflow after the user provides additional conditions.

In [4]:
user_query_2 = "Calculate adsorption in HKUST-1"
print(user_query_2)

Calculate adsorption in HKUST-1


In [5]:
demo2 = build_workflow(user_query_2, agents)
show_result("Initial parsed bundle", demo2["bundle"])

=== Parsed Queries ===
- HKUST-1-uptake: RASPAAgent → uptake (HKUST-1, guest=None)

Initial parsed bundle
{
  "queries": [
    {
      "Name": "HKUST-1-uptake",
      "Agent": "RASPAAgent",
      "Property": "uptake",
      "MOF": "HKUST-1",
      "Guest": null,
      "QueryText": "Calculate adsorption in HKUST-1"
    }
  ],
  "analysis_enabled": false,
  "simulation_input": {
    "present": false,
    "snippets": []
  },
  "simulation_input_status": "ok",
  "simulation_input_message": "",
  "needs_clarification": true,
  "missing_fields": [
    "property",
    "guest",
    "temperature",
    "pressure"
  ],
  "clarification_question": "Which adsorption property do you want for HKUST-1 (e.g., uptake, isotherm, Henry coefficient, binding energy, selectivity, diffusivity)? Also, please specify the guest molecule and the conditions: temperature and pressure (or a pressure range if you want an isotherm)."
}


In [7]:
print("needs_clarification:", demo2["bundle"].get("needs_clarification", False))
print("clarification_question:")
print(demo2["bundle"].get("clarification_question", "No clarification needed."))

needs_clarification: True
clarification_question:
Which adsorption property do you want for HKUST-1 (e.g., uptake, isotherm, Henry coefficient, binding energy, selectivity, diffusivity)? Also, please specify the guest molecule and the conditions: temperature and pressure (or a pressure range if you want an isotherm).


In [8]:
user_reply_2 = "Use CO2 at 298 K and 1 bar."

refined_query_2 = (
    user_query_2.strip()
    + "\n"
    + f"Additional user-provided conditions: {user_reply_2}"
)

print(refined_query_2)

Calculate adsorption in HKUST-1
Additional user-provided conditions: Use CO2 at 298 K and 1 bar.


In [10]:
demo2_refined = build_workflow(refined_query_2, agents)

show_result("Refined parsed bundle", demo2_refined["bundle"])
show_result("Refined parsed queries", demo2_refined["parsed"])
show_result("Refined planned workflows", demo2_refined["plans"])

=== Parsed Queries ===
- HKUST-1-CO2-uptake-1bar-298K: RASPAAgent → uptake (HKUST-1, guest=CO2)

Refined parsed bundle
{
  "queries": [
    {
      "Name": "HKUST-1-CO2-uptake-1bar-298K",
      "Agent": "RASPAAgent",
      "Property": "uptake",
      "MOF": "HKUST-1",
      "Guest": "CO2",
      "QueryText": "Calculate adsorption in HKUST-1\nAdditional user-provided conditions: Use CO2 at 298 K and 1 bar."
    }
  ],
  "analysis_enabled": false,
  "simulation_input": {
    "present": false,
    "snippets": []
  },
  "simulation_input_status": "ok",
  "simulation_input_message": "",
  "needs_clarification": true,
  "missing_fields": [
    "property"
  ],
  "clarification_question": "Which adsorption property would you like for CO2 in HKUST-1 at 298 K and 1 bar (e.g., single-point uptake, isotherm over a pressure range, Henry coefficient, diffusivity, binding energy)?"
}

Refined parsed queries
[
  {
    "Name": "HKUST-1-CO2-uptake-1bar-298K",
    "Agent": "RASPAAgent",
    "Property": "

In [11]:
# Uncomment the lines below if you want to run the full workflow.
# Depending on your environment, this may trigger the actual RASPA-based execution.

results_2 = await demo2_refined["worker"].run_async()
show_result("Execution results", results_2)

Saving output to: /home/users/skyljw0714/SimMOF/working_dir/HKUST-1_CO2_uptake
Successfully fetched HKUST-1 structure
[MOFCHECKER] Validation passed: HKUST-1.cif


/home/users/skyljw0714/anaconda3/envs/simmof/lib/python3.9/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 10020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


[RAG] RASPA model hints enabled
[RASPAErrorAgent] batch polling start: n=1, interval=20s, timeout=259200s
[RASPAErrorAgent] DONE and no error strings → OK.

=== RASPAOutputAgent: parsing output in /home/users/skyljw0714/SimMOF/working_dir/HKUST-1_CO2_uptake ===
[RASPAOutputAgent] uptake(excess) = 49.5188180899 [cm^3 (STP)/cm^3 framework] from output_HKUST-1_2.2.2_298.000000_100000.data

=== ResponseAgent: final user-facing answer ===

Computed CO2 adsorption in HKUST-1 at 298 K and 1 bar.  
The RASPA simulation completed successfully (parse status: ok).  
Simulated excess uptake: 49.52 cm^3 (STP)/cm^3 framework.  
If you need absolute uptake or a conversion (e.g., to mmol/g or wt%), I can provide that.

=== End of Response ===


Execution results
{
  "HKUST-1_CO2_uptake": {
    "HKUST-1_CO2_uptake_job": {
      "plan_name": "HKUST-1_CO2_uptake",
      "job_name": "HKUST-1_CO2_uptake",
      "job_id": "HKUST-1_CO2_uptake_job",
      "agent": "RASPAAgent",
      "mof": "HKUST-1",
      "

## Example 3 — Reusing user-provided simulation input (reproduce mode)

In this example, the user directly provides a simulation input snippet.

SimMOF will:
1. detect the provided input,
2. check whether it matches the requested task,
3. and allow the user to reuse it as-is.

This corresponds to a reproduce-style workflow,
where existing simulation inputs are reused instead of being regenerated.

In [4]:
user_query_3 = """I want to calculate CO2 uptake in UiO-66.

Here is my RASPA input:

SimulationType                MonteCarlo
NumberOfCycles               2000
NumberOfInitializationCycles 1000
PrintEvery                   100
Forcefield                   UFF
FrameworkName                UiO-66
ExternalTemperature          298.0
ExternalPressure             100000
Component 0 MoleculeName     CO2
            MoleculeDefinition TraPPE
            TranslationProbability 1.0
            RotationProbability    1.0
            ReinsertionProbability 1.0
"""
print(user_query_3)

I want to calculate CO2 uptake in UiO-66.

Here is my RASPA input:

SimulationType                MonteCarlo
NumberOfCycles               2000
NumberOfInitializationCycles 1000
PrintEvery                   100
Forcefield                   UFF
FrameworkName                UiO-66
ExternalTemperature          298.0
ExternalPressure             100000
Component 0 MoleculeName     CO2
            MoleculeDefinition TraPPE
            TranslationProbability 1.0
            RotationProbability    1.0
            ReinsertionProbability 1.0



In [5]:
demo3 = build_workflow(user_query_3, agents)

show_result("Parsed bundle", demo3["bundle"])
show_result("Parsed queries", demo3["parsed"])
show_result("Simulation input", demo3["simulation_input"])

=== Parsed Queries ===
- UiO-66-CO2-uptake: RASPAAgent → uptake (UiO-66, guest=CO2)

Parsed bundle
{
  "queries": [
    {
      "Name": "UiO-66-CO2-uptake",
      "Agent": "RASPAAgent",
      "Property": "uptake",
      "MOF": "UiO-66",
      "Guest": "CO2",
      "QueryText": "I want to calculate CO2 uptake in UiO-66.\n\nHere is my RASPA input:\n\nSimulationType                MonteCarlo\nNumberOfCycles               2000\nNumberOfInitializationCycles 1000\nPrintEvery                   100\nForcefield                   UFF\nFrameworkName                UiO-66\nExternalTemperature          298.0\nExternalPressure             100000\nComponent 0 MoleculeName     CO2\n            MoleculeDefinition TraPPE\n            TranslationProbability 1.0\n            RotationProbability    1.0\n            ReinsertionProbability 1.0\n"
    }
  ],
  "analysis_enabled": false,
  "simulation_input": {
    "present": true,
    "snippets": [
      {
        "software": "RASPA",
        "text": "Simulat

In [6]:
print("simulation_input_status:", demo3["bundle"]["simulation_input_status"])
print("simulation_input_message:")
print(demo3["bundle"]["simulation_input_message"] or "(no issues detected)")

simulation_input_status: needs_user_confirmation
simulation_input_message:
Your RASPA snippet targets CO2 in UiO-66, but it looks incomplete/risky for an uptake (GCMC) calculation and may yield zero or unreliable adsorption. Potential issues:
- No swap/insertion/deletion moves (GCMC) defined; only translation/rotation/reinsertion.
- Missing framework block (e.g., "Framework 0"), UnitCells, and FixedFramework settings.
- No VDW/coulomb cutoffs or Ewald/charge settings; TraPPE-CO2 uses charges.
- Framework charges not specified (UFF without charges is likely inaccurate for UiO-66).
- ExternalPressure alone may not set the chemical potential for GCMC.
Please reply with KEEP to proceed as-is, REGENERATE for a corrected GCMC input, or paste your corrected input.


In [7]:
user_decision_3 = "KEEP"   # or "REGENERATE" or a correction instruction
print("User decision:", user_decision_3)

User decision: KEEP


In [8]:
bundle3 = demo3["bundle"]

if bundle3["simulation_input_status"] == "needs_user_confirmation":
    if user_decision_3.upper() == "KEEP":
        print("Keeping the user-provided simulation input as-is.")
        results_3 = await demo3["worker"].run_async()
        show_result("Execution results", results_3)

    elif user_decision_3.upper() == "REGENERATE":
        print("Discarding the user-provided simulation input and regenerating from scratch.")

        regenerated_demo3 = build_workflow(
            demo3["bundle"]["queries"][0]["QueryText"],
            agents,
        )

        results_3 = await regenerated_demo3["worker"].run_async()
        show_result("Execution results", results_3)

    else:
        print("Applying a user-requested correction to the provided simulation input...")

        patched_simulation_input = patch_simulation_input_with_user_reply(
            simulation_input=bundle3["simulation_input"],
            user_reply=user_decision_3,
            parsed_queries=demo3["parsed"],
            llm=LLM_DEFAULT,
        )

        show_result("Patched simulation input", patched_simulation_input)
else:
    print("No confirmation needed. Running directly...")
    results_3 = await demo3["worker"].run_async()
    show_result("Execution results", results_3)

Keeping the user-provided simulation input as-is.
Saving output to: /home/users/skyljw0714/SimMOF/working_dir/UiO-66_CO2_uptake
Successfully fetched UiO-66 structure
[MOFCHECKER] Validation passed: UiO-66.cif
[RASPAErrorAgent] batch polling start: n=1, interval=20s, timeout=259200s
